This notebook provides a simple 6‑cell starter baseline for the CSIRO Biomass competition.
It includes data loading, preprocessing, a multi‑target regression model, training loop, and submission generation.
Designed for educational purposes — not guaranteed to reach medal level.
Participants are encouraged to adjust paths, parameters, augmentations, and train their own weights to improve performance.

In [ ]:
# =========================
# Cell 1 — Imports & Config
# =========================
import os, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import timm
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

CFG = {
    "input_dir": "/kaggle/input/csiro-biomass",  # adjust if needed
    "model_name": "tf_efficientnet_b4_ns",
    "img_size": 384,
    "batch_size": 32,
    "epochs": 3,          # short run for starter
    "lr": 2e-4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "targets_order": ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]
}

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

In [ ]:
# =========================
# Cell 2 — Load Data
# =========================
train = pd.read_csv(os.path.join(CFG["input_dir"], "train.csv"))
test  = pd.read_csv(os.path.join(CFG["input_dir"], "test.csv"))

# absolute paths
train["image_path"] = train["image_path"].apply(lambda p: os.path.join(CFG["input_dir"], p))
test["image_path"]  = test["image_path"].apply(lambda p: os.path.join(CFG["input_dir"], p))

# pivot train to wide format: one row per image with 5 targets
wide = train.pivot_table(index="image_path",
                         columns="target_name",
                         values="target",
                         aggfunc="mean").reset_index()

for t in CFG["targets_order"]:
    if t not in wide.columns:
        wide[t] = np.nan
wide = wide[["image_path"] + CFG["targets_order"]]
wide.fillna(0.0, inplace=True)

In [ ]:
# =========================
# Cell 3 — Dataset & Transforms
# =========================
train_tf = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(),
    ToTensorV2()
])

valid_tf = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.Normalize(),
    ToTensorV2()
])

class BiomassTrainDS(Dataset):
    def __init__(self, df, transforms):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.target_cols = CFG["targets_order"]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(image=img)["image"]
        y = row[self.target_cols].values.astype(np.float32)
        return img, torch.tensor(y)

class BiomassTestDS(Dataset):
    def __init__(self, df_unique_images, transforms):
        self.df = df_unique_images.reset_index(drop=True)
        self.transforms = transforms
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(image=img)["image"]
        return img, row["image_path"]


In [ ]:
# =========================
# Cell 4 — Model
# =========================
class BiomassModel(nn.Module):
    def __init__(self, model_name, n_outputs=5):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone.num_features, n_outputs)
        )
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)

model = BiomassModel(CFG["model_name"], n_outputs=len(CFG["targets_order"])).to(CFG["device"])


In [ ]:
# =========================
# Cell 5 — Training Loop (Skeleton)
# =========================
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
criterion = nn.SmoothL1Loss()

mask = np.random.rand(len(wide)) < 0.9
tr_df = wide[mask].copy()
va_df = wide[~mask].copy()

tr_loader = DataLoader(BiomassTrainDS(tr_df, train_tf),
                       batch_size=CFG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
va_loader = DataLoader(BiomassTrainDS(va_df, valid_tf),
                       batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

for ep in range(CFG["epochs"]):
    model.train()
    tr_losses = []
    for imgs, targets in tr_loader:
        imgs = imgs.to(CFG["device"])
        targets = targets.to(CFG["device"])
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        tr_losses.append(loss.item())
    model.eval()
    va_losses = []
    with torch.no_grad():
        for imgs, targets in va_loader:
            imgs = imgs.to(CFG["device"])
            targets = targets.to(CFG["device"])
            preds = model(imgs)
            loss = criterion(preds, targets)
            va_losses.append(loss.item())
    print(f"Epoch {ep+1}/{CFG['epochs']} | train {np.mean(tr_losses):.4f} | valid {np.mean(va_losses):.4f}")


In [ ]:
# =========================
# Cell 6 — Prediction & Submission
# =========================
unique_test_imgs = test[["image_path"]].drop_duplicates().reset_index(drop=True)
test_loader = DataLoader(BiomassTestDS(unique_test_imgs, valid_tf),
                         batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

model.eval()
image_to_pred = {}
with torch.no_grad():
    for imgs, paths in test_loader:
        imgs = imgs.to(CFG["device"])
        outputs = model(imgs).cpu().numpy()
        for p, out in zip(paths, outputs):
            image_to_pred[p] = out

pred_vals = []
for _, row in test.iterrows():
    img_path = row["image_path"]
    tname = row["target_name"]
    idx = CFG["targets_order"].index(tname)
    val = float(image_to_pred.get(img_path, np.zeros(len(CFG["targets_order"])))[idx])
    pred_vals.append(val)

submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "target": pred_vals
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved")



